# 15.3 - Copyright & Intellectual Property

**Module 15 - Ethics & Responsible AI** | Netsetos GenAI Engineering

This notebook turns copyright from a launch-day legal surprise into a set of engineering checks you run before you ship. Each cell is a small, keyless, library-free routine: quantify how much of your corpus is copyrighted, catch verbatim memorization in outputs, score the four fair-use factors, measure substantial similarity, verify licence compatibility across your stack, keep a provenance ledger for opt-outs, and gate the release on the conditions that keep your provider indemnity valid.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Environment prep. This lesson is entirely keyless - no API calls, no models, no external services. Every check below is plain Python on illustrative data, so the only thing to prepare is an optional install for anyone running in a fresh Colab.

> No API key or GPU needed - all cells run locally on standard Python.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A do-nothing setup cell. The only pip line is commented out (nothing in this lesson actually imports `openai`), and the comment notes that the worked examples use fixed, seeded values so results are reproducible.

**How the code works - step by step**
- **commented `pip install openai`** - present only for a fresh Colab; not required, since no cell makes an API call.
- **reproducibility note** - flags that the lesson uses fixed illustrative values throughout, so every run prints the same numbers.

**In one line:** nothing to install - the whole notebook is keyless local arithmetic.

**What you'll see:** (no output - environment prep)

## 1 - Quantify your training-data exposure

Risk starts with the training data. A frontier model learns from an enormous scraped corpus, and most of that text is copyrighted and unlicensed. You cannot settle in code whether training on it is fair use - courts are split - so you do the engineering thing and *measure* the exposure: what fraction of your corpus is copyright-encumbered? That number sizes the bet you are making and feeds every mitigation that follows.

In [ ]:
# THE TRAINING-DATA QUESTION: models learn from scraped text, most of it copyrighted. Whether that TRAINING is lawful is the
# central fight (NYT v OpenAI, Getty v Stability). Quantify the exposure first: how much of your corpus is copyright-encumbered?
corpus = {"copyrighted_scraped": 720, "licensed": 180, "public_domain": 100}   # documents, illustrative
total = sum(corpus.values())
print("Training corpus of {:,} documents:".format(total))
for k, v in corpus.items():
    print("  {:<22} {:>4}  ({:.0%})".format(k, v, v / total))
exposed = corpus["copyrighted_scraped"] / total
print()
print("{:.0%} of the corpus is copyrighted material used WITHOUT a licence. Two legal positions collide:".format(exposed))
print("  - developers argue TRAINING is transformative fair use (learning patterns, not copying expression);")
print("  - rightsholders argue it is mass infringement plus market harm. Courts are split; treat unlicensed scraping as a RISK,")
print("not a settled right. The mitigations in this lesson (memorization checks, licensing, provenance, indemnity) manage that risk.")

# Output:
# Training corpus of 1,000 documents:
#   copyrighted_scraped     720  (72%)
#   licensed                180  (18%)
#   public_domain           100  (10%)
#
# 72% of the corpus is copyrighted material used WITHOUT a licence. Two legal positions collide:
#   - developers argue TRAINING is transformative fair use (learning patterns, not copying expression);
#   - rightsholders argue it is mass infringement plus market harm. Courts are split; treat unlicensed scraping as a RISK,
# not a settled right. The mitigations in this lesson (memorization checks, licensing, provenance, indemnity) manage that risk.

**Explanation**

A measurement, not a model call: it takes a labelled breakdown of the corpus and computes the single number that matters - the share used without a licence. The point is that this is the input to risk management, not a legal answer; it tells you how big the exposure is, then prints the two opposing legal positions so you know why the number is only a risk figure.

**How the code works - step by step**
- **`corpus` dict** - three buckets of documents: 720 copyrighted-and-scraped, 180 licensed, 100 public-domain.
- **`total` / per-bucket loop** - sums to 1,000 and prints each bucket as a count and a percentage of the whole.
- **`exposed`** - divides copyrighted-scraped by total to get the headline exposure figure.
- **closing prints** - lays out the two legal positions (developers: transformative fair use; rightsholders: mass infringement plus market harm) to frame the number as a risk, not a settled right.

**In one line:** count the corpus buckets -> divide out the unlicensed share -> read it as the size of the risk.

**What you'll see:** It prints the 1,000-document breakdown (72% copyrighted, 18% licensed, 10% public domain), then reports that 72% of the corpus is copyrighted material used without a licence and summarizes the two colliding legal positions.

## 2 - Detect memorization with the longest shared run

Training on a work is a grey area; emitting it *verbatim* is far clearer infringement. Extraction attacks have shown large models can be induced to regurgitate memorized training text, so you add an output-side check - a plagiarism detector for your own model. The reliable signal is the longest run of consecutive shared words between an output and a candidate source: a short overlap is chance, a long one is copying.

In [ ]:
# MEMORIZATION / REGURGITATION: training on a work is contested; reproducing it VERBATIM is clearer infringement. Detect it by
# the longest run of shared words between an output and a source - a plagiarism check for model outputs. No library needed.
source = "It was the best of times it was the worst of times it was the age of wisdom".split()
output = "In summary it was the best of times it was the worst of times as Dickens wrote".split()
def longest_common_run(a, b):
    best = 0
    for i in range(len(a)):
        for j in range(len(b)):
            k = 0
            while i + k < len(a) and j + k < len(b) and a[i + k] == b[j + k]:
                k += 1
            best = max(best, k)
    return best
run = longest_common_run(source, output)
THRESHOLD = 6                                       # a run this long is very unlikely by chance -> memorization
print("longest verbatim shared run: {} words.".format(run))
print("  '{}'".format(" ".join(output[output.index("it"):output.index("it") + run]) if "it" in output else ""))
print("threshold = {} words -> {}".format(THRESHOLD, "MEMORIZATION flagged (likely reproduces a protected expression)" if run >= THRESHOLD else "no memorization"))
print()
print("Training on data is arguable; emitting a long verbatim copy is not. Run this check on outputs and block or filter the ones")
print("that reproduce a protected passage - the same defence that keeps you out of the NYT-style 'the model quoted our article' claim.")

# Output:
# longest verbatim shared run: 11 words.
#   'it was the best of times it was the worst of'
# threshold = 6 words -> MEMORIZATION flagged (likely reproduces a protected expression)
#
# Training on data is arguable; emitting a long verbatim copy is not. Run this check on outputs and block or filter the ones
# that reproduce a protected passage - the same defence that keeps you out of the NYT-style 'the model quoted our article' claim.

**Explanation**

A pure string-processing filter with no library: it slides every start position of the output against every start position of the source, extends each match word-by-word, and keeps the longest run. Read it as a plagiarism check you point at your model's outputs before they ship - if the longest verbatim run clears a threshold, you block the output.

**How the code works - step by step**
- **`source` / `output`** - two word lists that share a long verbatim stretch in the middle.
- **`longest_common_run(a, b)`** - double loop over start indices; the inner `while` extends a match as long as words stay equal, tracking the best length found.
- **`THRESHOLD = 6`** - the run length above which a shared passage is treated as memorization rather than coincidence.
- **comparison and prints** - reports the run length, shows the shared phrase, and flags the output when `run >= THRESHOLD`.

**In one line:** find the longest word-for-word overlap -> if it clears the threshold, flag it as a reproduced passage.

**What you'll see:** It reports the longest verbatim shared run is 11 words, prints the shared phrase ('it was the best of times it was the worst of'), and because 11 exceeds the 6-word threshold flags the output as MEMORIZATION to block or filter.

## 3 - Score the four fair-use factors

When a use is neither clearly infringing nor clearly licensed, US law resolves it with fair use, and fair use has a structure: 17 U.S.C. 107 lists four factors a court weighs - purpose, nature, amount, and market effect. The crucial thing for an engineer is that this is a *balancing* test, not a checklist: you do not win by counting factors in your favour, and market harm (the fourth factor) has carried the most weight in recent AI rulings.

In [ ]:
# THE FAIR-USE FOUR FACTORS (17 U.S.C. 107): US courts weigh four factors. Score each from -2 (against fair use) to +2 (for it),
# sum, and read the lean. It is a BALANCING test, not a checklist - one strong factor (market harm) can sink an otherwise-fair use.
FACTORS = ["purpose (transformative? non-commercial?)", "nature (factual? published?)",
           "amount (how much, and the heart of the work?)", "market effect (a substitute?)"]
def fair_use(scores):
    total = sum(scores)
    lean = "likely FAIR USE" if total >= 2 else ("likely INFRINGING" if total <= -2 else "genuinely uncertain")
    return total, lean
research = [2, 1, 1, 2]      # a transformative research tool that does not substitute for the original
substitute = [-1, -1, 0, -2] # a commercial product that reproduces enough to replace buying the original
for name, s in [("research/transformative use", research), ("commercial substitute", substitute)]:
    total, lean = fair_use(s)
    print("{}:".format(name))
    for f, v in zip(FACTORS, s):
        print("  {:+d}  {}".format(v, f))
    print("  -> total {:+d}  ->  {}\n".format(total, lean))
print("The 4th factor (market harm) carries the most weight in recent AI rulings. 'Transformative' helps but does not automatically win.")

# Output:
# research/transformative use:
#   +2  purpose (transformative? non-commercial?)
#   +1  nature (factual? published?)
#   +1  amount (how much, and the heart of the work?)
#   +2  market effect (a substitute?)
#   -> total +6  ->  likely FAIR USE
#
# commercial substitute:
#   -1  purpose (transformative? non-commercial?)
#   -1  nature (factual? published?)
#   +0  amount (how much, and the heart of the work?)
#   -2  market effect (a substitute?)
#   -> total -4  ->  likely INFRINGING
#
# The 4th factor (market harm) carries the most weight in recent AI rulings. 'Transformative' helps but does not automatically win.

**Explanation**

A scoring harness that models the balancing test: each factor gets a score from -2 (against fair use) to +2 (for it), the scores are summed, and the total is read as a lean toward fair, infringing, or uncertain. It is deliberately a judgement aid, not a formula - the scores are inputs a human assigns, and the code just totals them and reads the balance.

**How the code works - step by step**
- **`FACTORS` list** - the four statutory factors as human-readable labels.
- **`fair_use(scores)`** - sums the four scores and maps the total to a lean: `>= 2` likely fair, `<= -2` likely infringing, otherwise genuinely uncertain.
- **`research` / `substitute`** - two example score vectors: a transformative research tool `[2,1,1,2]` and a commercial substitute `[-1,-1,0,-2]`.
- **loop and prints** - for each use, lists every factor with its score, then prints the total and the lean; a closing line notes market harm is the heaviest factor.

**In one line:** score each of the four factors, sum, and read the lean - remembering market harm weighs most.

**What you'll see:** It prints both uses factor-by-factor: the research/transformative tool totals +6 -> likely FAIR USE, the commercial substitute totals -4 -> likely INFRINGING, and closes by noting the fourth factor (market harm) carries the most weight.

## 4 - Measure substantial similarity

Memorization catches the exact copy; infringement does not require word-for-word copying. An output can be *substantially similar* to a protected work while changing the words, and still infringe. The governing line is the idea-expression distinction: ideas are free, the specific expression is protected. You approximate this by measuring the overlap of distinctive content and comparing it to the de-minimis level courts tolerate.

In [ ]:
# SUBSTANTIAL SIMILARITY: even a NON-verbatim output can infringe if it is substantially similar to a protected work. Measure the
# overlap of distinctive content (Jaccard over meaningful tokens) and compare to the de minimis line courts tolerate.
def tokens(s):
    stop = {"the", "a", "an", "of", "and", "to", "in", "is", "it", "was", "as"}
    return set(w.lower().strip(".,") for w in s.split() if w.lower() not in stop)
original = "A young wizard with a lightning scar attends a secret school of magic and fights a dark lord"
para_a = "A teenage sorcerer bearing a lightning scar enrolls at a hidden magic school to battle a dark lord"  # close paraphrase
para_b = "A detective in a rainy city solves a murder using forensic science and sharp deduction"               # unrelated
def jaccard(x, y):
    a, b = tokens(x), tokens(y)
    return round(len(a & b) / len(a | b), 3)
DE_MINIMIS = 0.20
for name, text in [("close paraphrase", para_a), ("unrelated work", para_b)]:
    sim = jaccard(original, text)
    verdict = "SUBSTANTIALLY SIMILAR (infringement risk)" if sim >= 0.30 else ("de minimis / OK" if sim < DE_MINIMIS else "gray zone - review")
    print("{:<18} similarity {:.3f}  ->  {}".format(name, sim, verdict))
print()
print("Verbatim copying is the easy case; substantial similarity catches the paraphrase that keeps the protected EXPRESSION. Ideas")
print("are free, expression is protected - so a similar plot is fine, but reusing the distinctive wording and details is the risk.")

# Output:
# close paraphrase   similarity 0.316  ->  SUBSTANTIALLY SIMILAR (infringement risk)
# unrelated work     similarity 0.000  ->  de minimis / OK
#
# Verbatim copying is the easy case; substantial similarity catches the paraphrase that keeps the protected EXPRESSION. Ideas
# are free, expression is protected - so a similar plot is fine, but reusing the distinctive wording and details is the risk.

**Explanation**

A similarity harness built on a Jaccard overlap of meaningful tokens: it strips common stopwords, turns each text into a set of distinctive words, and computes the intersection-over-union against a protected original. The score is a heuristic flag for the idea-expression line, not a legal verdict - it surfaces the paraphrase that kept too much protected expression while letting a merely shared theme through.

**How the code works - step by step**
- **`tokens(s)`** - lowercases, strips punctuation, and removes a stopword set so only distinctive content words remain, returned as a set.
- **`original` / `para_a` / `para_b`** - a protected sentence, a close paraphrase of it, and an unrelated sentence.
- **`jaccard(x, y)`** - intersection size over union size of the two token sets, rounded to three places.
- **`DE_MINIMIS = 0.20`, thresholds** - classifies each score: `>= 0.30` substantially similar, `< 0.20` de minimis, in between a gray zone to review.
- **loop and prints** - reports each paraphrase's similarity and verdict, then notes ideas are free but distinctive expression is the risk.

**In one line:** overlap the distinctive words -> compare to the de-minimis and substantial lines -> flag what keeps the protected expression.

**What you'll see:** It prints the close paraphrase at similarity 0.316 -> SUBSTANTIALLY SIMILAR (infringement risk), and the unrelated work at 0.000 -> de minimis / OK, then contrasts free ideas with protected expression.

## 5 - Check licence compatibility across the stack

A GenAI product is an assembly - a base model, fine-tuning data, eval sets, libraries - and every one carries a licence, and they do not all combine. Permissive licences (MIT, Apache, CC-BY) combine freely; copyleft (GPL) infects a derivative and can force your whole product open-source; non-commercial (CC-BY-NC) forbids selling; proprietary data cannot be redistributed. One incompatible licence deep in the stack can block a launch.

In [ ]:
# LICENSE COMPATIBILITY: your model, its training data, and its dependencies each carry a licence, and they do not all combine.
# Copyleft (GPL) "infects" a derivative work; permissive (MIT/Apache) does not. Check before you ship, not after.
# category: permissive (combine freely), copyleft (forces the whole work open), proprietary (cannot redistribute), noncommercial
LICENSE = {"MIT": "permissive", "Apache-2.0": "permissive", "GPL-3.0": "copyleft",
           "CC-BY": "permissive", "CC-BY-NC": "noncommercial", "proprietary": "proprietary"}
def can_ship_commercial_closed(components):
    problems = []
    for name, lic in components:
        cat = LICENSE[lic]
        if cat == "copyleft": problems.append("{} ({}) forces your whole product open-source".format(name, lic))
        if cat == "noncommercial": problems.append("{} ({}) forbids commercial use".format(name, lic))
        if cat == "proprietary": problems.append("{} ({}) cannot be redistributed".format(name, lic))
    return problems
stack = [("base model", "Apache-2.0"), ("fine-tune data", "CC-BY"), ("a helper lib", "GPL-3.0"), ("eval set", "CC-BY-NC")]
print("shipping a closed-source commercial product built from:")
for name, lic in stack:
    print("  {:<16} {:<12} ({})".format(name, lic, LICENSE[lic]))
problems = can_ship_commercial_closed(stack)
print()
print("verdict: {}".format("CLEAR to ship" if not problems else "BLOCKED - {} conflict(s):".format(len(problems))))
for p in problems:
    print("   - " + p)
print("License incompatibility is a supply-chain bug: one GPL dependency or one non-commercial dataset can force-open or sink a product.")

# Output:
# shipping a closed-source commercial product built from:
#   base model       Apache-2.0   (permissive)
#   fine-tune data   CC-BY        (permissive)
#   a helper lib     GPL-3.0      (copyleft)
#   eval set         CC-BY-NC     (noncommercial)
#
# verdict: BLOCKED - 2 conflict(s):
#    - a helper lib (GPL-3.0) forces your whole product open-source
#    - eval set (CC-BY-NC) forbids commercial use
# License incompatibility is a supply-chain bug: one GPL dependency or one non-commercial dataset can force-open or sink a product.

**Explanation**

A supply-chain checker: it maps each component's licence to a category, then walks your stack collecting the categories that are incompatible with shipping a closed-source commercial product. Read it like a dependency vulnerability scan - it returns a list of blocking conflicts (empty means clear), so you catch the problem in CI rather than in a lawsuit.

**How the code works - step by step**
- **`LICENSE` dict** - maps each licence name to a category: permissive, copyleft, noncommercial, or proprietary.
- **`can_ship_commercial_closed(components)`** - for each component, flags a problem when its category is copyleft (forces open-source), noncommercial (forbids selling), or proprietary (cannot redistribute); returns the list of problems.
- **`stack`** - the four components: Apache-2.0 model, CC-BY data, GPL-3.0 helper lib, CC-BY-NC eval set.
- **prints** - lists each component with its category, then the verdict - CLEAR if no problems, otherwise BLOCKED with each conflict spelled out.

**In one line:** categorize each component's licence -> collect the copyleft/non-commercial/proprietary conflicts -> block if any remain.

**What you'll see:** It lists the four components with their licence categories, then prints 'BLOCKED - 2 conflict(s)': the GPL-3.0 helper lib would force the product open-source and the CC-BY-NC eval set forbids commercial use.

## 6 - Keep a provenance ledger for the TDM opt-out

The 2026 rules added a machine-level obligation on top of the fair-use debate. Under the EU DSM Directive Article 4, rightsholders can reserve their works from text-and-data mining - a machine-readable opt-out - and if a source opted out you must exclude it. The EU AI Act Article 53 layers on a training-data summary requirement. Both share one engineering prerequisite: per-document provenance recorded at ingestion.

In [ ]:
# PROVENANCE + THE TDM OPT-OUT: 2026 rules (EU AI Act Art. 53 + DSM Directive Art. 4) let rightsholders RESERVE their work from
# text-and-data mining (an opt-out, e.g. in robots.txt or a header). If a source opted out, you must EXCLUDE it - so track provenance.
sources = [   # (source, licence, opted_out_of_TDM)
    ("openwebtext dump", "unknown", False),
    ("nytimes.com articles", "copyrighted", True),     # reserved rights -> must exclude
    ("wikipedia", "CC-BY-SA", False),
    ("licensed news partner", "commercial licence", False),
    ("artiststation portfolios", "copyrighted", True)]  # opted out
print("data provenance ledger (source, licence, TDM opt-out):")
must_exclude, unknown = [], []
for src, lic, opt in sources:
    flag = "EXCLUDE (rights reserved)" if opt else ("verify licence" if lic == "unknown" else "OK to use")
    print("  {:<26} {:<20} {}".format(src, lic, flag))
    if opt: must_exclude.append(src)
    if lic == "unknown": unknown.append(src)
print()
print("{} source(s) reserved rights (opt-out) and must be removed from training; {} have unknown licences to verify.".format(len(must_exclude), len(unknown)))
print("You cannot honour an opt-out or answer 'what were you trained on?' (EU AI Act Art. 53 training-data summary) without a")
print("provenance ledger. Provenance is the foundation - keep it per document, at ingestion, or you cannot comply after the fact.")

# Output:
# data provenance ledger (source, licence, TDM opt-out):
#   openwebtext dump           unknown              verify licence
#   nytimes.com articles       copyrighted          EXCLUDE (rights reserved)
#   wikipedia                  CC-BY-SA             OK to use
#   licensed news partner      commercial licence   OK to use
#   artiststation portfolios   copyrighted          EXCLUDE (rights reserved)
#
# 2 source(s) reserved rights (opt-out) and must be removed from training; 1 have unknown licences to verify.
# You cannot honour an opt-out or answer 'what were you trained on?' (EU AI Act Art. 53 training-data summary) without a
# provenance ledger. Provenance is the foundation - keep it per document, at ingestion, or you cannot comply after the fact.

**Explanation**

A ledger walk, not a model call: it iterates a per-source record of (source, licence, opted-out?) and classifies each row into exclude, verify, or OK. The point it drives home is that you can only honour an opt-out or produce an Art. 53 summary if this ledger was built at ingestion time - reconstructing 'what were we trained on?' after the fact across billions of documents is impossible.

**How the code works - step by step**
- **`sources` list** - five tuples of (source, licence, opted_out_of_TDM), two of which reserved their rights.
- **loop over sources** - computes a flag per row: EXCLUDE if opted out, 'verify licence' if the licence is unknown, otherwise OK to use; collects the exclude and unknown lists.
- **prints** - renders the ledger as a table, then counts how many sources must be removed and how many need licence verification.
- **closing lines** - tie the exclusions to the EU AI Act Art. 53 training-data summary and stress provenance must be per-document, at ingestion.

**In one line:** walk the provenance ledger -> flag opted-out sources to EXCLUDE and unknown licences to verify.

**What you'll see:** It prints the five-row ledger, marking the NYTimes articles and artist portfolios as EXCLUDE (rights reserved) and the openwebtext dump as 'verify licence', then reports 2 sources must be removed and 1 has an unknown licence.

## 7 - Gate the release and check indemnity

The final step turns the previous six into one decision: does this release ship? Providers now offer copyright indemnity (Microsoft Copilot Copyright Commitment, OpenAI Copyright Shield), but every one is conditional on using the built-in guardrails. So you build a compliance gate that mirrors those conditions - and remember that a pure-AI output has no human author and therefore no copyright in the US, so you cannot stop others copying it.

In [ ]:
# THE COMPLIANCE GATE + INDEMNITY: providers now offer copyright INDEMNITY (Microsoft Copilot Copyright Commitment, Google, OpenAI
# Copyright Shield) - but ONLY if you use the guardrails. Gate the deploy on them, and know that AI-only outputs are not copyrightable.
CHECKS = {                                          # each must hold for the deploy to ship and the indemnity to apply
    "provenance ledger complete": True,
    "TDM opt-outs excluded": True,
    "output memorization filter on": True,
    "licence conflicts resolved": False,            # a GPL dependency is unresolved (from step 5)
    "human authorship on published outputs": True}  # US Copyright Office: pure-AI output has no copyright
def gate(checks):
    return [name for name, ok in checks.items() if not ok]
failures = gate(CHECKS)
print("COPYRIGHT COMPLIANCE GATE (model release):")
for name, ok in CHECKS.items():
    print("  [{}] {}".format("x" if ok else " ", name))
print()
print("gate: {}".format("PASS - ship; provider indemnity applies" if not failures else "BLOCK - {} unmet, indemnity VOID:".format(len(failures))))
for f in failures:
    print("   - " + f)
print("Indemnity is conditional: skip the guardrails and you lose the coverage AND take the liability. And remember - a pure-AI")
print("output is not copyrightable (US Copyright Office), so you cannot stop others from copying it; only human-authored work is.")

# Output:
# COPYRIGHT COMPLIANCE GATE (model release):
#   [x] provenance ledger complete
#   [x] TDM opt-outs excluded
#   [x] output memorization filter on
#   [ ] licence conflicts resolved
#   [x] human authorship on published outputs
#
# gate: BLOCK - 1 unmet, indemnity VOID:
#    - licence conflicts resolved
# Indemnity is conditional: skip the guardrails and you lose the coverage AND take the liability. And remember - a pure-AI
# output is not copyrightable (US Copyright Office), so you cannot stop others from copying it; only human-authored work is.

**Explanation**

A release gate: a dict of named conditions (each true or false) that must all hold for the deploy to ship and the provider indemnity to apply. It collects the unmet checks and blocks on any failure - the same way a CI gate fails safe on a failing test. The teaching point is that indemnity is insurance with conditions, so one skipped guardrail both stops the ship and voids the coverage.

**How the code works - step by step**
- **`CHECKS` dict** - the five conditions (provenance complete, opt-outs excluded, memorization filter on, licence conflicts resolved, human authorship), one of which - licence conflicts, carried over from step 5 - is False.
- **`gate(checks)`** - returns the names of every condition that is not met.
- **prints** - renders each check as a `[x]`/`[ ]` line, then the verdict: PASS (ship, indemnity applies) if nothing failed, otherwise BLOCK with the unmet checks and 'indemnity VOID'.
- **closing lines** - note indemnity is conditional on the guardrails, and that a pure-AI output is not copyrightable under US Copyright Office guidance.

**In one line:** check every release condition -> block on any failure, which also voids the conditional provider indemnity.

**What you'll see:** It prints the five-item checklist with 'licence conflicts resolved' unchecked, then 'BLOCK - 1 unmet, indemnity VOID', and closes by noting a pure-AI output has no copyright so you cannot stop others copying it.

Copyright in GenAI is not one legal question but a pipeline of engineering checks: size the exposure, filter memorization, weigh fair use, measure similarity, verify licences, keep provenance, and gate the release. Each cell here is deliberately keyless arithmetic on illustrative facts - the point is the shape of the control, not the exact numbers, and every threshold is a heuristic that surfaces risk for a human to judge, never a legal verdict. Next, Lesson 15.4 takes up what the model learned about specific people (privacy), and 15.5 covers proving how media was made (provenance and watermarking).